# 3 · Forecast flux

The encoder carries a small **normalizing-flow head** trained to predict flux a short
horizon away from each timestep. `predict_flux` reproduces that forecast: for each target
point it conditions on the encoder's forward state a few steps *before* and its backward
state a few steps *after*, then draws from the flow to produce a **median prediction** and a
**16th–84th percentile band** (the model's predictive uncertainty at that horizon).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import encotess

# Same style of toy light curve as tutorial 1.
rng = np.random.default_rng(1)
n = 4000
time = (np.arange(n) * (2.0 / 60.0 / 24.0)).astype(np.float32)
flux = 0.8 * np.sin(2 * np.pi * time / 2.5) + rng.normal(0, 1.0, n)
flux = ((flux - flux.mean()) / flux.std()).astype(np.float32)
flux_err = np.ones(n, dtype=np.float32)

meta = {'cadence_s': 120.0, 'Tmag': 10.5, 'sector': 45, 'camera': 1, 'ccd': 2,
        'parallax': 5.0, 'parallax_error': 0.02, 'G0': 10.8, 'G0_err': 0.01,
        'BPRP0': 1.1, 'BPRP0_err': 0.03, 'median_flux': 1e4, 'iqr_half_flux': 50.0}

enc = encotess.load_encoder(device='cpu')

### Predict

`offset` is the forecast horizon in timesteps (it must satisfy `2 * offset < len(flux)`);
`n_samples` sets how many flow draws form the median and the band. The result is a dict of
`flux` (median), `p16`, `p84`, and the target `time` axis.

In [ ]:
pred = encotess.predict_flux(enc, flux, flux_err, time, metadata=meta,
                             offset=8, n_samples=64)

print('predicted points:', len(pred['flux']))
print('median flux[:5] :', pred['flux'][:5].round(3))
print('p16    flux[:5] :', pred['p16'][:5].round(3))
print('p84    flux[:5] :', pred['p84'][:5].round(3))

### Plot the forecast against the observed flux

A short window makes the median trace and the p16–p84 band easy to see.

In [ ]:
# Align the observed flux to the prediction's target axis (predict_flux trims the
# edges and the horizon; here it drops `offset` points from each end).
k = 8
obs_flux = flux[k:len(flux) - k]     # matches pred['time'] within the trimmed interior
t = pred['time']

w = slice(0, 400)                    # zoom into the first ~0.5 day
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t[w], obs_flux[w], '.', ms=3, alpha=0.5, label='observed')
ax.plot(t[w], pred['flux'][w], '-', lw=1.5, label='predicted median')
ax.fill_between(t[w], pred['p16'][w], pred['p84'][w], alpha=0.3, label='p16–p84 band')
ax.set_xlabel('time [days]'); ax.set_ylabel('normalized flux')
ax.set_title('EncoTESS flux forecast (horizon = 8 steps)')
ax.legend(); plt.tight_layout(); plt.show()

### Notes

- The band widens where the model is less certain and is narrowest where the light curve is
  most predictable — it is genuine predictive spread from the flow, not a fitted error bar.
- The horizon (`offset`) is a knob: larger horizons ask the model to predict further ahead
  and generally yield wider bands.
- On a pure-noise light curve the median collapses toward zero and the band reflects the
  point-to-point scatter, since there is no structure to forecast.